# Tutorial 07: Hologram Generation and Artifacts

This notebook isolates the final stage of the CK notebook: Jones propagation, ideal holograms, detector holograms, helicity differences/sums, reconstructions, and detector artifacts.

The first part creates a lightweight synthetic hologram so that users can explore post-processing instantly. The second part contains a small full simulation that can be enabled when they want to run the whole physical chain.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    pass
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator


## 1. Helper plotting function

In [ ]:
def show_image(ax, data, title, cmap="viridis", percentile=True):
    if percentile:
        vmin, vmax = np.nanpercentile(data, [1, 99])
    else:
        vmin, vmax = None, None
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    return im


## 2. Synthetic CR/CL holograms for instant artifact exploration

This section is intentionally not a physical simulation. It creates a plausible-looking diffraction pattern, a weak magnetic contrast term, a beamstop, Poisson/readout noise, saturation, and missing-pixel stripes. It teaches what the downstream containers do before users run the slower Jones propagation.


In [ ]:
rng = np.random.default_rng(4)
shape = (512, 512)
y, x = np.indices(shape)
cy, cx = np.array(shape) // 2
r = np.hypot(y - cy, x - cx)
theta = np.arctan2(y - cy, x - cx)

speckle = np.abs(np.fft.fftshift(np.fft.fft2(rng.normal(size=shape)))) ** 2
speckle = speckle / speckle.max()
radial_envelope = 1 / (1 + (r / 45) ** 3)
charge_hologram = 2e4 * radial_envelope * (0.25 + speckle)
magnetic_contrast = 700 * radial_envelope * np.sin(7 * theta + 0.07 * r)

ideal_CR = charge_hologram + magnetic_contrast
ideal_CL = charge_hologram - magnetic_contrast

# Beamstop mask: 1 means blocked in this tutorial cell.
beamstop = (r < 32).astype(float)
beamstop[:, cx - 3:cx + 3] = np.maximum(beamstop[:, cx - 3:cx + 3], (r[:, cx - 3:cx + 3] < 220))

def detect_synthetic(ideal, seed):
    local_rng = np.random.default_rng(seed)
    photons = np.clip(ideal * (1 - beamstop), 0, None)
    noisy = local_rng.poisson(photons)
    noisy = noisy + local_rng.normal(loc=30, scale=5, size=shape)
    noisy = np.clip(noisy, 0, 25000)
    noisy[120:124, :] = 0      # dead row block
    noisy[:, 390:394] = 0      # dead column block
    return noisy

detected_CR = detect_synthetic(ideal_CR, 10)
detected_CL = detect_synthetic(ideal_CL, 11)

hologram_config = sim.HologramConfig(
    ideal_holograms={"CR": ideal_CR, "CL": ideal_CL},
    detected_holograms={"CR": detected_CR, "CL": detected_CL},
    exit_waves={
        "CR": np.sqrt(np.clip(ideal_CR, 0, None)).astype(complex),
        "CL": np.sqrt(np.clip(ideal_CL, 0, None)).astype(complex),
    },
)
hologram_config.compute_differences()
hologram_config.compute_sums()
hologram_config.compute_reconstructions()


## 3. Visualize ideal, detected, sum, and difference holograms

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
items = [
    (hologram_config.ideal_holograms["CR"], "ideal CR"),
    (hologram_config.ideal_holograms["CL"], "ideal CL"),
    (hologram_config.ideal_holograms["sum"], "ideal CR + CL"),
    (hologram_config.ideal_holograms["diff"], "ideal CR - CL"),
    (hologram_config.detected_holograms["CR"], "detected CR"),
    (hologram_config.detected_holograms["CL"], "detected CL"),
    (hologram_config.detected_holograms["sum"], "detected CR + CL"),
    (hologram_config.detected_holograms["diff"], "detected CR - CL"),
]
for ax, (data, title) in zip(axes.flat, items):
    show_image(ax, data, title, cmap="magma")


## 4. Fourier transform holography reconstruction

The reconstruction is `fftshift(fft2(fftshift(hologram)))`. Use the difference channel for magnetic contrast and the sum channel for charge/background structure.


In [ ]:
hologram_config.visualize_reconstruction(source="detected", helicity="diff")


## 5. Turn individual synthetic artifacts on/off

Use this cell to teach the visual effect of each detector artifact separately.


In [ ]:
artifact_variants = {
    "ideal": ideal_CR,
    "beamstop only": ideal_CR * (1 - beamstop),
    "poisson only": rng.poisson(np.clip(ideal_CR, 0, None)),
    "beamstop + noise + dead pixels": detected_CR,
}

fig, axes = plt.subplots(1, len(artifact_variants), figsize=(14, 3.4))
for ax, (title, data) in zip(axes, artifact_variants.items()):
    show_image(ax, data, title, cmap="magma")


## 6. Optional full physical simulation

Set `RUN_FULL_SIMULATION = True` to run the complete miniature chain:

1. X-ray and detector setup.
2. Beamstop creation.
3. Sample material stack.
4. Magnetic pattern and holography mask.
5. Dielectric tensor construction.
6. Gaussian illumination.
7. Jones propagation for `CR` and `CL`.
8. Ideal and detected holograms, including detector noise and beamstop.

The defaults are intentionally small compared with CK, but this can still take time.


In [ ]:
RUN_FULL_SIMULATION = False


In [ ]:
if RUN_FULL_SIMULATION:
    detector_shape = (96, 96)
    detector_center = tuple(np.array(detector_shape) // 2)
    oversampling = 2

    xray_config = sim.XRayConfig(
        energy=778.0,
        photon_flux=5e8,
        pol="CR",
        coherence_length=(10e-6, 10e-6),
    )
    xray_config.setup()

    beamstop_config = sim.BeamstopConfig(
        bs_method="circular",
        bs_detector_distance=0.010,
        bs_center=detector_center,
        bs_config={
            "radius": 180e-6,
            "sigma": 10e-6,
            "wire_width": 40e-6,
            "angle": np.deg2rad(25),
            "antialias": 3,
            "seed": 4,
        },
    )
    detector_config = sim.DetectorConfig(
        shape=detector_shape,
        pixel_size=20e-6,
        sample_to_detector_distance=0.075,
        detector_center=detector_center,
        # Detector response; counts_per_photon and readout noise are canonical here.
        detector_params={
            "readout_noise_average": 20,
            "readout_noise_sigma": 3,
            "detector_threshold": 5e4,
            "counts_per_photon": 100,
            "quantum_efficiency": 0.9,
            "noise_seed": 20,
        },
        # Acquisition timing/frame settings are a separate stage.
        measurement_config={
            "exposure_time": 1.0,
            "number_frames": 1,
            "max_counts_per_image": None,
        },
        beamstop_config=beamstop_config,
    ignore_flat_detector_curvature=False,
    )
    detector_config.setup()
    real_space_pixel_size = detector_config.calc_realspace_resolution(xray_config.beam_params) / oversampling

    sample_shape = [0, oversampling * detector_shape[0], oversampling * detector_shape[1]]
    sample_config = sim.SampleConfig(
        recipe="Au(80)/Cr(5)/SiN(80)/Pt(4)Co(6)/Pt(2)",
        sample_shape=sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        xray_config=xray_config,
        sample_name="mini hologram tutorial",
    )
    sample_config.setup()

    magnetic_pattern_config = sim.MagneticPatternConfig(
        pattern_type_method="binary_labyrinth_pattern",
        shape=tuple(sample_shape[1:]),
        real_space_pixel_size=real_space_pixel_size,
        pattern_config={
            "stripe_width": 60e-9,
            "sigma": 4e-9,
            "n_steps": 60,
            "seed": 3,
            "use_gpu": False,
        },
    )
    magnetic_pattern_config.create_pattern()
    mz = magnetic_pattern_config.magnetic_pattern
    magnetization = pattern_generator.map_magnetization_to_3d(
        magnetic_pattern_x=np.zeros_like(mz),
        magnetic_pattern_y=np.sqrt(np.clip(1 - np.abs(mz) ** 2, 0, 1)),
        magnetic_pattern_z=mz,
        nr_repeats=sample_config.sample_structure.sample_shape[0],
    )
    sample_config.assign_magnetic_pattern(magnetization)

    layer_thicknesses = sample_config.sample_structure.layer_thicknesses
    membrane_index = sample_config.sample_structure.layer_names.index("SiN")
    aperture_config = sim.FrontApertureConfig(
        aperture_method="FTH_circular",
        aperture_shape=tuple(sample_shape),
        real_space_pixel_size=real_space_pixel_size,
        aperture_thicknesses=layer_thicknesses,
        aperture_layer_names=sample_config.sample_structure.layer_names,
        aperture_config={
            "apertures_type": ["OH", "RH", "RH"],
            "apertures_radius": [555e-9, 100e-9, 34e-9],
            "apertures_center": [(0.0, 0.0), (-1240e-9, -1225e-9), (1225e-9, -1200e-9)],
            "apertures_sigma": [4e-9, 2e-9, 2e-9],
            "apertures_angle": [0.0, 0.0, 0.0],
            "apertures_ellipticity": [1.0, 1.0, 1.0],
            "apertures_roughness": [0.0, 0.02, 0.02],
            "apertures_roughness_modes": [(0, 0), (3, 10), (3, 10)],
            "apertures_seed": [1, 2, 3],
            "apertures_top_radius_factor": [1.3, 1.5, 1.75],
            "aperture_taper_depth": float(np.sum(layer_thicknesses[:max(0, membrane_index - 2)])),
            "thickness_OH": float(np.sum(layer_thicknesses[:membrane_index])),
        },
        use_roi=True,
    )
    aperture_config.setup()
    sample_config.assign_aperture_mask(aperture_config.return_aperture())
    illumination_alpha_beam = (0.0, 0.0)  # rad (alpha_y, alpha_x)
    contrast_beam_direction = sim.light_beam.beam_direction_from_alpha(illumination_alpha_beam)
    if np.allclose(contrast_beam_direction, [0.0, 0.0, 1.0], atol=1e-14, rtol=0.0):
        contrast_beam_direction = None
    sample_config.sample_structure.calculate_final_dielectric_tensor(
        use_aperture_roi=True,
        compact=True if contrast_beam_direction is None else False,
        beam_direction=contrast_beam_direction,
    )

    illumination_config = sim.IlluminationConfig(
        XRayConfig=xray_config,
        shape=tuple(sample_shape[1:]),
        real_space_pixel_size=real_space_pixel_size,
        illumination_function="gaussian",
        illumination_config={
            "center": np.array([0.0, 0.0]),
            "distance": 1e-3,
            "fwhm": 0.45e-6,
            "alpha_beam": (0.0, 0.0),  # rad (alpha_y, alpha_x)
        },
    )
    illumination_config.setup()

    full_holograms = sim.HologramConfig(
        sample_x=sample_config.sample_structure.x,
        sample_y=sample_config.sample_structure.y,
        detector_layout=detector_config.detector_layout,
    )
    propagator_method = "Jones"  # Change to "Scalar" for the scalar refractive-index propagator.

    for pol_index, pol in enumerate(["CR", "CL"]):
        detector_config.detector_params["noise_seed"] = 20 + pol_index
        illumination_config.update_polarization(pol)

        propagator_config = sim.SamplePropagatorConfig(
            SampleConfig=sample_config,
            IlluminationConfig=illumination_config,
            propagator_method=propagator_method,
            propagator_config={
                "propagator_method": propagator_method,
                "propagate": False,
                "jones_apply_zero_order_phase": True,
                "multislice_propagation_roi": True,
                # ROI padding gives local diffraction corrections room to taper smoothly back to the exp(-1j*k0*dz) baseline.
                "multislice_propagation_roi_padding_px": 12,
                # Overlapping padded ROI crops are merged before local propagation.
                "multislice_propagation_roi_merge_overlaps": True,
            },
        )
        propagator_config.setup()

        detector_config.assign_propagated_wavefront(propagator_config)
        detector_config.detect_hologram()

        full_holograms.add_exit_waves({pol: propagator_config.return_scalar_wavefield()})
        full_holograms.add_holograms({pol: detector_config.return_ideal_hologram()}, source="ideal")
        full_holograms.add_holograms(
            {pol: detector_config.return_detected_hologram(store_no_beamstop=True)},
            source="detected",
        )
        full_holograms.add_holograms(
            {pol: detector_config.return_detected_hologram_without_beamstop()},
            source="detected_no_beamstop",
        )

    full_holograms.compute_differences()
    full_holograms.compute_sums()
    full_holograms.compute_reconstructions()
    full_holograms.visualize_averages()


## 7. Full-simulation reconstruction

Run this after enabling the full simulation above.


In [ ]:
if RUN_FULL_SIMULATION:
    full_holograms.visualize_reconstruction(source="detected", helicity="diff")
    full_holograms.visualize_reconstruction(source="detected", helicity="sum")
else:
    print("Full simulation is disabled. Set RUN_FULL_SIMULATION = True above and rerun the full-simulation cells.")
